<a href="https://colab.research.google.com/github/kasinadhsarma1/Notebooks/blob/main/bilingual_transcribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bilingual (Hindi + English) call transcription

Runs OpenAI Whisper `large-v3` on a free Colab GPU to produce:
1. The **original-language transcript** (Hindi/English as actually spoken)
2. An **English translation** of the same audio

then pairs them line-by-line into a bilingual `.txt` per file (and a combined file).

**Before running:** Runtime menu -> Change runtime type -> select a GPU (T4 is fine, has ~15GB VRAM, plenty for large-v3).

In [1]:
!nvidia-smi

Mon Sep 21 10:56:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Upload your mp3 files

Run this cell and use the file picker to upload all 10 (or however many) mp3 files.

In [3]:
from google.colab import files
import os

os.makedirs("audio", exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    os.rename(name, os.path.join("audio", name))
print("Uploaded:", os.listdir("audio"))

Saving 0f39e6f2-a9e0-4523-87e6-f43910c919c1.mp3 to 0f39e6f2-a9e0-4523-87e6-f43910c919c1 (1).mp3
Saving 1ed0c766-6621-4e46-87f9-95992209223d.mp3 to 1ed0c766-6621-4e46-87f9-95992209223d (1).mp3
Saving 3c8573c5-0564-4b80-a4f4-09cbe18fd2c5.mp3 to 3c8573c5-0564-4b80-a4f4-09cbe18fd2c5 (1).mp3
Saving 935afa38-43b0-4c14-8238-7c1745f11513.mp3 to 935afa38-43b0-4c14-8238-7c1745f11513 (1).mp3
Saving 7102fc1c-050f-48e0-b171-7f2403300cba.mp3 to 7102fc1c-050f-48e0-b171-7f2403300cba (1).mp3
Saving a5cbacf6-fd97-4976-9833-f6f130b74cd4.mp3 to a5cbacf6-fd97-4976-9833-f6f130b74cd4 (1).mp3
Saving bfacbd70-36cb-4ffd-b6cb-801fc4e4b063.mp3 to bfacbd70-36cb-4ffd-b6cb-801fc4e4b063 (1).mp3
Uploaded: ['935afa38-43b0-4c14-8238-7c1745f11513 (1).mp3', '7102fc1c-050f-48e0-b171-7f2403300cba (1).mp3', 'bfacbd70-36cb-4ffd-b6cb-801fc4e4b063 (1).mp3', 'a5cbacf6-fd97-4976-9833-f6f130b74cd4 (1).mp3', '3c8573c5-0564-4b80-a4f4-09cbe18fd2c5 (1).mp3', '0f39e6f2-a9e0-4523-87e6-f43910c919c1 (1).mp3', '1ed0c766-6621-4e46-87f9-9599

### (Alternative) Mount Google Drive instead of uploading

If your mp3s are already in Drive, skip the upload cell above and use this instead — point `AUDIO_DIR` at the Drive folder.

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')
# AUDIO_DIR = "/content/drive/MyDrive/path/to/your/mp3s"
AUDIO_DIR = "audio"

## 2. Load the model (large-v3 on GPU)

In [5]:
import whisper

model = whisper.load_model("large-v3")  # downloads ~3GB once, then cached

100%|██████████████████████████████████████| 2.88G/2.88G [00:28<00:00, 109MiB/s]


## 3. Transcribe (original language) + Translate (English), per file

In [7]:
import glob, json, os

os.makedirs("out", exist_ok=True)
mp3_files = sorted(glob.glob(os.path.join(AUDIO_DIR, "*.mp3")))
print(f"Found {len(mp3_files)} files")

results = {}
for path in mp3_files:
    name = os.path.splitext(os.path.basename(path))[0]
    print("Processing:", name)

    orig = model.transcribe(path, task="transcribe")   # original language (Hindi script for Hindi speech)
    en   = model.transcribe(path, task="translate")    # English translation

    results[name] = {"orig": orig, "en": en}

    with open(f"out/{name}.orig.json", "w", encoding="utf-8") as f:
        json.dump(orig, f, ensure_ascii=False, indent=2)
    with open(f"out/{name}.en.json", "w", encoding="utf-8") as f:
        json.dump(en, f, ensure_ascii=False, indent=2)

print("Done.")

Found 10 files
Processing: 0f39e6f2-a9e0-4523-87e6-f43910c919c1 (1)
Processing: 1ed0c766-6621-4e46-87f9-95992209223d (1)
Processing: 3c8573c5-0564-4b80-a4f4-09cbe18fd2c5 (1)
Processing: 7102fc1c-050f-48e0-b171-7f2403300cba (1)
Processing: 935afa38-43b0-4c14-8238-7c1745f11513 (1)
Processing: a5cbacf6-fd97-4976-9833-f6f130b74cd4 (1)
Processing: bfacbd70-36cb-4ffd-b6cb-801fc4e4b063 (1)
Processing: c69ddd23-e495-42a3-a268-dc33247e4723
Processing: d7aad2ad-9e68-47a0-8f8a-f74ac2782563
Processing: e10e735c-9170-4a0f-ae08-7196b1fb9290 (2)
Done.


## 4. Build bilingual line-by-line transcripts

Pairs each original-language segment with the closest English segment by timestamp overlap (segmentation between the two passes can differ slightly, so exact index-pairing isn't reliable).

In [8]:
def best_match(seg, other_segments):
    """Find the other-language segment with the most time overlap."""
    s0, s1 = seg["start"], seg["end"]
    best, best_overlap = None, 0.0
    for o in other_segments:
        overlap = max(0.0, min(s1, o["end"]) - max(s0, o["start"]))
        if overlap > best_overlap:
            best, best_overlap = o, overlap
    return best

os.makedirs("bilingual", exist_ok=True)
combined_lines = []

for name, r in results.items():
    orig_segs = r["orig"]["segments"]
    en_segs = r["en"]["segments"]

    lines = [f"=== {name} ===", ""]
    for seg in orig_segs:
        match = best_match(seg, en_segs)
        orig_text = seg["text"].strip()
        en_text = match["text"].strip() if match else ""
        lines.append(f"[{seg['start']:.1f}s] {orig_text}")
        if en_text:
            lines.append(f"    EN: {en_text}")
        lines.append("")

    text = "\n".join(lines)
    with open(f"bilingual/{name}.bilingual.txt", "w", encoding="utf-8") as f:
        f.write(text)
    combined_lines.append(text)

with open("bilingual/combined.bilingual.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(combined_lines))

print("Bilingual transcripts written to ./bilingual/")

Bilingual transcripts written to ./bilingual/


## 5. Download the results

In [9]:
import shutil
shutil.make_archive("bilingual_transcripts", "zip", "bilingual")

from google.colab import files as gfiles
gfiles.download("bilingual_transcripts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>